# CNN Image Classification Benchmark

This notebook benchmarks TensorFlow/Keras CNN models for two tasks: binary smile classification and six-class sign-language digit classification. It uses local HDF5 datasets under `../datasets/` and saves benchmark artifacts under `../outputs/`.

## 1. Environment Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.data_loader import load_happy_dataset, load_signs_dataset, print_dataset_summary, MissingDatasetError
from src.models import build_smile_baseline, build_smile_improved_cnn, build_smile_augmented_cnn, build_signs_baseline, build_signs_improved_cnn, build_signs_augmented_cnn
from src.benchmark import run_benchmark
from src.visualization import show_image_samples

config.set_global_determinism(config.SEED)
config.ensure_output_dirs()
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory: {config.DATA_DIR}")

## 2. Dataset Loading and Inspection

The project expects `train_happy.h5`, `test_happy.h5`, `train_signs.h5`, and `test_signs.h5` under `datasets/`. If the files are unavailable, the notebook stops gracefully after printing the missing-file message.

In [ ]:
try:
    happy_data = load_happy_dataset(config.DATA_DIR)
    signs_data = load_signs_dataset(config.DATA_DIR)
    DATA_AVAILABLE = True
except MissingDatasetError as error:
    DATA_AVAILABLE = False
    print(error)

In [ ]:
if DATA_AVAILABLE:
    print("Happy House dataset")
    print_dataset_summary(happy_data, "smile")
    print("
SIGNS dataset")
    print_dataset_summary(signs_data, "signs")

In [ ]:
if DATA_AVAILABLE:
    show_image_samples(happy_data.x_train, happy_data.y_train, max_images=5)
    show_image_samples(signs_data.x_train, signs_data.y_train, max_images=5)

## 3. Baseline CNN Models

In [ ]:
if DATA_AVAILABLE:
    smile_baseline = build_smile_baseline(input_shape=happy_data.x_train.shape[1:])
    signs_baseline = build_signs_baseline(input_shape=signs_data.x_train.shape[1:], num_classes=signs_data.y_train.shape[1])
    smile_baseline.summary()
    signs_baseline.summary()

## 4. Improved CNN Models

In [ ]:
if DATA_AVAILABLE:
    smile_improved = build_smile_improved_cnn(input_shape=happy_data.x_train.shape[1:])
    signs_improved = build_signs_improved_cnn(input_shape=signs_data.x_train.shape[1:], num_classes=signs_data.y_train.shape[1])
    smile_augmented = build_smile_augmented_cnn(input_shape=happy_data.x_train.shape[1:])
    signs_augmented = build_signs_augmented_cnn(input_shape=signs_data.x_train.shape[1:], num_classes=signs_data.y_train.shape[1])
    print("Smile improved parameters:", smile_improved.count_params())
    print("Smile augmented parameters:", smile_augmented.count_params())
    print("SIGNS improved parameters:", signs_improved.count_params())
    print("SIGNS augmented parameters:", signs_augmented.count_params())

## 5. Training and Evaluation

Run the full controlled benchmark. To shorten a quick local smoke test, pass a small value such as `epochs=2`. For final experiments, run `python -m src.benchmark` from the repository root.

In [ ]:
if DATA_AVAILABLE:
    # For a quick notebook smoke test, set epochs=2.
    # For full training, remove the epochs override or run: python -m src.benchmark
    results_df = run_benchmark(epochs=2)
else:
    results_df = pd.DataFrame()

## 6. Benchmark Results

In [ ]:
if not results_df.empty:
    display(results_df)
else:
    print("Benchmark results will be available after the local datasets are added and the benchmark is run.")

## 7. Visual Analysis

Training curves, confusion matrices, and the aggregate benchmark comparison are saved under `outputs/figures/`.

In [ ]:
figures_dir = config.FIGURES_DIR
if figures_dir.exists():
    for path in sorted(figures_dir.glob("*.png")):
        print(path.relative_to(PROJECT_ROOT))

## 8. Summary of Findings

Use the generated benchmark table and figures to compare the baseline, improved CNN, and augmented CNN variants. Report final conclusions from the produced metrics only.